# 🚀 Pelatihan Model Deteksi Objek Kustom (YOLO)

Notebook ini dirancang untuk alur kerja lengkap: dari konversi label segmentasi ke bounding box, konfigurasi dataset, pelatihan, evaluasi, hingga ekspor model untuk inferensi.

**Fitur:**
- Konversi otomatis Poligon → Bounding Box (Format YOLO)
- Pelatihan dengan augmentasi
- Evaluasi metrik (mAP, Precision, Recall, FPS)
- Ekspor ke ONNX/TensorRT

## Langkah 0: Pembuatan Data Dummy (Opsional tapi Direkomendasikan)
*Jalankan cell ini terlebih dahulu untuk membuat data tiruan. Ini memastikan seluruh pipeline di bawah ini berjalan tanpa error `FileNotFoundError` sebelum Anda menggunakan dataset asli Anda.*

In [ ]:
import os
import cv2
import numpyا.numpy as np
import shutil

# 1. Buat struktur direktori
dirs = [
    '/content/dataset/images/train',
    '/content/dataset/images/val',
    '/content/dataset/images/test',
    '/content/dataset/labels_seg_raw'
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# 2. Generate 15 gambar dan label segmentasi dummy
for i in range(15):
    # Gambar random noise
    img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
    cv2.imwrite(f'/content/dataset/images/train/dummy_{i}.jpg', img)
    
    # Label segmentasi dummy (class 0, dengan 4 titik poligon normalisasi)
    # Format YOLO Segmentasi: class_id x1 y1 x2 y2 x3 y3 x4 y4
    with open(f'/content/dataset/labels_seg_raw/dummy_{i}.txt', 'w') as f:
        f.write("0 0.2 0.2 0.8 0.2 0.8 0.8 0.2 0.8\n")

# 3. Pindahkan sebagian kecil ke folder val dan test untuk simulasi
shutil.copy('/content/dataset/images/train/dummy_14.jpg', '/content/dataset/images/val/dummy_14.jpg')
shutil.copy('/content/dataset/labels_seg_raw/dummy_14.txt', '/content/dataset/labels_seg_raw/dummy_14_val.txt')

shutil.copy('/content/dataset/images/train/dummy_13.jpg', '/content/dataset/images/test/dummy_13.jpg)
shutil.copy('/content/dataset/labels_seg_raw/dummy_13.txt', '/content/dataset/labels_seg_raw/dummy_13_test.txt')

print("✅ Data dummy berhasil dibuat di /content/dataset/")

## Langkah 1: Verifikasi GPU & Instalasi Lingkungan
Memastikan akselerasi GPU tersedia dan menginstal pustaka yang diperlukan.

In [ ]:
# Cek ketersediaan dan tipe GPU
!nvidia-smi

# Instalasi pustaka utama
# Catatan: ultralytics versi terbaru sudah mencakup YOLOv8/v11. 
!pip install -q ultralytics opencv-python matplotlib pyyaml

## Langkah 2: Skrip Konversi Label Segmentasi (Poligon) ke Bounding Box
Skrip ini membaca format segmentasi poligon `[class_id, x1, y1, x2, y2, ...]` dan mengubahnya secara otomatis menjadi format deteksi bounding box `[class_id, x_center, y_center, width, height]` menggunakan nilai ekstremum.

In [ ]:
import os
import glob

def convert_polygon_to_bbox(label_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    label_files = glob.glob(os.path.join(label_dir, "*.txt"))
    
    print(f"Ditemukan {len(label_files)} file label untuk dikonversi...")
    
    for file_path in label_files:
        filename = os.path.basename(file_path)
        out_path = os.path.join(output_dir, filename)
        
        with open(file_path, 'r') as f_in, open(out_path, 'w') as f_out:
            for line in f_in:
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                
                class_id = parts[0]
                coords = [float(x) for x in parts[1:]]
                
                # Ekstrak koordinat x dan y (x di indeks genap, y di indeks ganjil)
                xs = coords[0::2]
                ys = coords[1::2]
                
                # Hitung ekstremum
                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)
                
                # Konversi ke format YOLO Bounding Box
                x_center = (x_min + x_max) / 2.0
                y_center = (y_min + y_max) / 2.0
                width = x_max - x_min
                height = y_max - y_min
                
                f_out.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
    print("✅ Konversi label selesai!")

# Jalankan konversi untuk data train dan val
convert_polygon_to_bbox('/content/dataset/labels_seg_raw', '/content/dataset/labels/train')
# Jika ada folder val raw, jalankan juga untuk val:
# convert_polygon_to_bbox('/content/dataset/labels_seg_raw_val', '/content/dataset/labels/val')

## Langkah 3: Konfigurasi File Dataset (`dataset.yaml`)
Mendefinisikan struktur direktori dan nama kelas untuk dibaca oleh framework YOLO.

In [ ]:
import yaml

# Tentukan struktur direktori dataset
dataset_config = {
    'path': '/content/dataset',  # Root direktori dataset
    'train': 'images/train',     # Path relatif data latih
    'val': 'images/val',         # Path relatif data validasi
    'test': 'images/test',       # Path relatif data uji
    'names': {                   # Daftar kelas kustom (Sesuaikan dengan dataset Anda)
        0: 'objek_target_1',
        1: 'objek_target_2'
    }
}

# Simpan ke file yaml
with open('dataset.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("✅ File dataset.yaml berhasil dibuat.")
!cat dataset.yaml

## Langkah 4: Pelatihan Model
Melatih model YOLO dengan data kustom. 
*Catatan: Gunakan `yolov8n.pt` atau `yolov11n.pt` sebagai baseline. Ganti menjadi `yolo26n.pt` HANYA jika Anda memiliki file bobot kustom tersebut.*

In [ ]:
from ultralytics import YOLO

# Memuat arsitektur YOLO (Gunakan 'yolo26n.pt' jika tersedia, jika tidak, 'yolov8n.pt' adalah pilihan aman)
model_name = 'yolov8n.pt' # <-- Ganti ke 'yolo26n.pt' jika model kustom Anda sudah tersedia
model = YOLO(model_name)

print(f"Memulai pelatihan dengan {model_name}...")

# Menjalankan proses pelatihan kustom
results = model.train(
    data='dataset.yaml',
    epochs=50,          # Dikurangi menjadi 50 untuk demo data dummy. Ubah ke 100+ untuk data asli.
    imgsz=640,
    batch=16,
    workers=2,
    device=0,                   # Menggunakan GPU CUDA
    optimizer='AdamW',
    lr0=0.01,
    augment=True,               # Mengaktifkan fungsi augmentasi data
    project='/content/YOLO_Custom',
    name='train_run_1',
    save=True
)

## Langkah 5: Pengujian dan Evaluasi Performa
Mengevaluasi model terbaik (`best.pt`) pada data uji independen dan mengukur latensi inferensi.

In [ ]:
import time
import os

# Memuat bobot terbaik hasil pelatihan
best_model_path = '/content/YOLO_Custom/train_run_1/weights/best.pt'

if os.path.exists(best_model_path):
    best_model = YOLO(best_model_path)
    
    # Evaluasi metrik utama pada data uji independen
    print("\n--- Melakukan Evaluasi pada Data Test ---")
    metrics = best_model.val(data='dataset.yaml', split='test')
    
    print(f"mAP@0.5      : {metrics.box.map50:.4f}")
    print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
    print(f"Precision    : {metrics.box.mp:.4f}")
    print(f"Recall       : {metrics.box.mr:.4f}")
    
    # Pengukuran latensi inferensi
    print("\n--- Mengukur Latensi Inferensi ---")
    test_images = '/content/dataset/images/test'
    
    start_time = time.time()
    # Inference pada folder
    results = best_model(test_images, device=0, verbose=False)
    end_time = time.time()
    
    # Hitung jumlah gambar yang diproses
    total_images = len([name for name in os.listdir(test_images) if name.endswith(('.jpg', '.jpeg', '.png'))])
    
    if total_images > 0:
        avg_latency_ms = ((end_time - start_time) / total_images) * 1000
        fps = 1000 / avg_latency_ms
        
        print(f"Total Gambar   : {total_images}")
        print(f"Rata-rata Latensi : {avg_latency_ms:.2f} ms per citra")
        print(f"Kecepatan (FPS)   : {fps:.2f} FPS")
    else:
        print("Tidak ada gambar di folder test untuk diukur latensinya.")
else:
    print(f"⚠️ File model tidak ditemukan di: {best_model_path}")

## Langkah 6: Optimasi & Ekspor Model ke Mesin Inferensi (ONNX & TensorRT)
Mengekspor model ke format yang lebih ringan dan cepat untuk deployment (Edge Devices, ROS, atau NVIDIA Jetson).

In [ ]:
import os

best_model_path = '/content/YOLO_Custom/train_run_1/weights/best.pt'

if os.path.exists(best_model_path):
    model_for_export = YOLO(best_model_path)
    
    print("\n📦 Mengekspor ke format ONNX (dengan kuantisasi FP16)...")
    onnx_path = model_for_export.export(
        format='onnx',
        half=True,          # Kuantisasi FP16 untuk akselerasi
        dynamic=False,      # Ukuran input statis untuk latensi optimal
        simplify=True       # Menyederhanakan grafik ONNX
    )
    print(f"✅ ONNX berhasil diekspor ke: {onnx_path}")
    
    # Ekspor ke format TensorRT Engine (Opsional, memerlukan lingkungan NVIDIA GPU)
    # print("\n📦 Mengekspor ke format TensorRT Engine...")
    # engine_path = model_for_export.export(format='engine', half=True, device=0)
    # print(f"✅ TensorRT berhasil diekspor ke: {engine_path}")
else:
    print("⚠️ Model belum dilatih atau tidak ditemukan. Lewati langkah ekspor.")